# **Jobs Preprocessing**

In [ ]:
# !pip install neo4j
# !pip install graphdatascience
# !pip install sentence-transformers
# !pip install torch torch-geometric pandas
# !pip install pandas


In [ ]:
import pandas as pd

df = pd.read_excel('/content/Emma_first_batch.xlsx')
df.head()

# Extracting Experience Level

In [ ]:
import re

def detect_level_from_title(title):
    if not isinstance(title, str):
        return None

    title = title.lower()

    if re.search(r"\bintern(ship)?\b", title):
        return "intern"

    elif re.search(r"\bfresher\b", title):
        return "fresher"

    elif re.search(r"\b(junior|jr|entry|entry-level|associate|graduate)\b", title):
        return "junior"

    elif re.search(r"\b(mid|mid-level)\b", title):
        return "mid"

    elif re.search(r"\b(senior|sr|senior-level)\b", title):
        return "senior"

    elif re.search(r"\b(lead|principal|staff|architect|expert)\b", title):
        return "lead"

    return None

In [ ]:
def map_experience_level(years):
    if pd.isna(years):
        return "unspecified"
    elif years < 1:
        return "fresher"
    elif years < 3:
        return "junior"
    elif years < 5:
        return "mid"
    elif years < 10:
        return "senior"
    else:
        return "lead"

In [ ]:
def extract_experience_level(title, years):

    # Extract both independently
    level_from_years = map_experience_level(years)

    level_from_title = detect_level_from_title(title)

    # Combine logic (title has priority)
    if level_from_title is not None:
        return level_from_title

    if level_from_years is not None:
        return level_from_years

    return "unspecified"

In [ ]:
df["final_level"] = df.apply(
    lambda row: extract_experience_level(row["title"], row["experience_level"]),
    axis=1
)

In [ ]:
df.head()

# Normalizing Job Title

In [ ]:
noise_words = {
    "junior", "jr", "senior", "sr", "lead", "principal", "pr",
    "intern", "internship", "remote", "onsite", "hybrid",
    "hiring", "urgent", "needed", "contract", "staff", "chief",
    "level", "i", "ii", "iii", "iv", "v", "1", "2", "3", "4", "5",
    "fresher", "co-op"
}


import pandas as pd
import re

def process_job_title(title):
    title_lower = str(title).lower()

    # 1. Aggressive Regex pre-cleaning
    title_lower = re.sub(r'\(.*?\)', '', title_lower)

    # Clean up punctuation
    base_title = re.sub(r'[^\w\s]', ' ', title_lower)
    base_title = re.sub(r'\s+', ' ', base_title).strip()

    words = base_title.split()
    # 3. Hierarchy Mapping (Most Specific -> Most Broad)

    # --- HIGH SPECIFICITY BUCKETS (Must go first to avoid traps) ---

    if any(word in base_title for word in ['devsecops', 'dev sec ops', 'dev security']):
        final_title = 'DevSecOps Engineer'
    elif any(word in base_title for word in ['network security', 'cyber', 'pentest', 'vulnerability', 'infosec', 'iam ']):
        final_title = 'Security Engineer'
    elif any(word in base_title for word in ['database', 'dba']):
        final_title = 'Database Administrator'
    elif any(word in base_title for word in ['game', 'unity', 'gamedev', 'unreal', 'world of warcraft']):
        final_title = 'Game Developer'
    elif any(word in base_title for word in ['rpa', 'robotics automation']):
        final_title = 'RPA Engineer'
    elif any(word in base_title for word in ['service now', 'servicenow', 'service management']):
        final_title = 'ServiceNow Engineer'
    elif any(word in base_title for word in ['sap', 'abap', 's4hana', 'fiori']):
        final_title = 'SAP Consultant'
    elif any(word in base_title for word in ['blockchain', 'crypto', 'ethereum', 'solidity', 'smart contract']):
        final_title = 'Blockchain Developer'

    elif any(word in words for word in ['ux', 'ui', 'cad', '3d']) or any(word in base_title for word in ['user experience', 'interaction design', 'modeler', 'animator']):
        final_title = 'UI/UX Designer'

    # --- MEDIUM SPECIFICITY BUCKETS ---

    elif any(word in base_title for word in ['devops', 'sre', 'reliability', 'kubernetes', 'openshift', 'elastic', 'kibana', 'ci cd']):
        final_title = 'DevOps/Platform Engineer'
    elif any(word in base_title for word in ['cloud', 'aws', 'azure', 'gcp']):
        final_title = 'Cloud Engineer'
    elif any(word in base_title for word in ['mobile', 'ios', 'android', 'swift', 'react native', 'flutter']):
        final_title = 'Mobile Developer'
    elif any(word in base_title for word in ['front end', 'frontend', 'ui ux', 'react', 'angular', 'javascript', 'drupal']):
        final_title = 'Frontend Developer'
    elif any(word in base_title for word in ['machine learning', 'ai', 'analytics', 'sql', 'prompt', 'data', 'nlp', 'computer vision', 'algorithm']):
        final_title = 'Data/AI Engineer'
    elif any(word in base_title for word in ['qa', 'test', 'quality', 'automation', 'sdet', 'validation', 'verification']):
        final_title = 'QA Engineer'
    elif any(word in base_title for word in ['embedded', 'cnc', 'firmware', 'hardware', 'robotics', 'bsp', 'fpga', 'asic', 'calibration']):
        final_title = 'Embedded/Hardware Engineer'
    elif any(word in base_title for word in ['product', 'scrum master', 'agile']):
        final_title = 'Product/Agile Manager'
    elif any(word in base_title for word in ['manager', 'director', 'head', 'chief', 'officer', 'vp']):
        final_title = 'Engineering Manager'
    elif any(word in base_title for word in ['operations', 'system administrator', 'sysadmin', 'server', 'linux', 'mac os', 'windows']):
        final_title = 'Operations/IT Engineer'
    elif any(word in base_title for word in ['network', 'cisco', 'telecommunications']):
        final_title = 'Network Engineer'
    elif any(word in base_title for word in ['field', 'sales', 'support', 'help desk', 'customer', 'technician']):
        final_title = 'Field/Support Engineer'
    elif any(word in base_title for word in ['mechanical', 'civil', 'structural', 'hvac', 'aerospace', 'project engineer', 'aviation', 'traffic', 'piping', 'construction']):
        final_title = 'Traditional Engineering (Mech/Civil)'
    elif any(word in base_title for word in ['bioinformatics', 'biotech', 'genomics', 'proteomics', 'computational biology']):
        final_title = 'Bioinformatics Scientist'


    # --- BROAD CATCH-ALLS ---
    elif 'full stack' in base_title or 'fullstack' in base_title:
        final_title = 'Full Stack Developer'
    elif any(word in base_title for word in ['back end', 'backend', 'java', 'python', 'c#', 'c++', 'golang', 'rust', 'net', 'php', 'node', 'api', 'middleware', 'integration']):
        final_title = 'Backend Developer'
    elif any(word in base_title for word in ['software', 'developer', 'development', 'programmer', 'sde', 'mts', 'member of technical staff']):
        final_title = 'Software Engineer'
    elif 'engineer' in base_title:
        final_title = 'General Engineer'
    else:
        # Fallback for weird titles (like "Plumber" or "CEO")
        final_title = 'Other'

    return pd.Series( final_title)

# df['Clean_Title'] = df['title'].apply(process_job_title)

# Extracting List of Skills

In [ ]:
# !pip install -q groq

In [ ]:
from groq import Groq
from google.colab import userdata

GROQ_API_KEY = userdata.get('GROQ_API_KEY')

client = Groq(api_key=GROQ_API_KEY)

In [ ]:
def get_completion(prompt, model="llama-3.1-8b-instant"):
    """
    A simple helper to send prompts to Groq LLaMA 3.1 models.
    Default: llama-3.1-8b-instant (fast & free)
    Other options:
        - llama-3.1-70b-versatile
        - llama-3.1-405b-reasoning (strong thinking)
    """
    chat_completion = client.chat.completions.create(
        model=model,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": prompt}]
    )
    return chat_completion.choices[0].message.content


In [ ]:
# Schema for parsing a Job Description (JD)
from google.genai import types

JD_SCHEMA = types.Schema(
    type=types.Type.OBJECT,
    properties={
        "job_title": types.Schema(type=types.Type.STRING, description="The official title of the role being advertised."),
        "company": types.Schema(type=types.Type.STRING, description="The name of the hiring company."),
        "min_experience_years": types.Schema(type=types.Type.NUMBER, description="The minimum required years of experience. Use null if unspecified"),
        "required_skills": types.Schema(type=types.Type.ARRAY, items=types.Schema(type=types.Type.STRING), description="A list of ESSENTIAL technical and soft skills."),
        "preferred_skills": types.Schema(type=types.Type.ARRAY, items=types.Schema(type=types.Type.STRING), description="A list of optional or 'nice-to-have' skills."),
        "qualifications": types.Schema(type=types.Type.ARRAY, items=types.Schema(type=types.Type.STRING), description="Required degrees or certifications (e.g., 'BS in Computer Science')."),
    }
)

In [ ]:
import json
#run on the first 100 rows only to test
df['skills'] = None
df['experience_level'] = None

for i, row in df.iloc[487:].iterrows():
    print("Processing row ", {i})
    text = row["description"]
    prompt = f"""
    Analyze the following {JD_SCHEMA} text and extract the information based *strictly* on the provided JSON schema.

    Skills must be concrete technologies such as programming languages, frameworks, tools, APIs, databases, or platforms.

    - **Skills:** Clean and standardize skills by:
        1. Removing duplicates.
        2. Capitalizing properly.
        3. Removing extra words.
        4. Keeping only core technical or professional skills.
        5. Convert phrases into common industry skill names.
        6. Keep programming languages **short and standard** (e.g., "C++", not "C++ Programming Language"; "Python", not "Python (Programming Language)").


     Return ONLY a valid JSON object matching this schema.

    {JD_SCHEMA} Text:
    ---
    {text}
    """
    response = get_completion(prompt)

    try:
        data = json.loads(response)
        skills = data.get("required_skills", []) + data.get("preferred_skills", [])
        skills = list(dict.fromkeys(skills))
        df.at[i, "skills"] = ", ".join(skills)
        df.at[i, 'experience_level'] = data.get("min_experience_years", 0)

        print("row", i, ": ", skills, "experience: ", data.get("min_experience_years"))
    except json.JSONDecodeError:
        print(f"Row {i} failed JSON parsing")
        df.at[i, 'skills'] = []
        df.at[i, 'experience_level'] = 0

In [ ]:
# Step 1: Initial split from the raw ',' separated string
df['Skill_List'] = df['skills'].str.split(',')
df = df.explode('Skill_List')
df['Skill_List'] = df['Skill_List'].str.strip()


In [ ]:
df.head(10)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 32405 entries, 0 to 1820
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                32251 non-null  object 
 1   title             32405 non-null  object 
 2   description       32405 non-null  object 
 3   clean_title       32405 non-null  object 
 4   skills            32405 non-null  object 
 5   experience_level  29103 non-null  float64
 6   final_level       32405 non-null  object 
 7   Skill_List        32405 non-null  object 
dtypes: float64(1), object(7)
memory usage: 2.2+ MB


In [ ]:
df.to_csv("/content/job_dataset.csv", index=False)

# KG Implementation

In [ ]:
# !pip install neo4j

In [ ]:
df2 = pd.read_csv('/content/job_dataset.csv')
df2.head()

In [ ]:
import pandas as pd
from neo4j import GraphDatabase
import os

# --- Configuration ---
URI = os.getenv("NEO4J_URI")
USERNAME = os.getenv("NEO4J_USERNAME")
PASSWORD = os.getenv("NEO4J_PASSWORD")
AUTH = (USERNAME, PASSWORD)
FILE_PATH = "/content/job_dataset.csv"
jobs = pd.read_csv(FILE_PATH)

driver = GraphDatabase.driver(URI, auth=AUTH)

def create_job_graph(tx, job_title, experience_level, skill):
    # Merge a unique Job node per title + experience_level
    tx.run(
        """
        MERGE (j:Job {title: $job_title, experience_level: $experience_level})
        MERGE (s:Skill {name: $skill})
        MERGE (j)-[:REQUIRES_SKILL]->(s)
        """,
        job_title=job_title,
        experience_level=experience_level,
        skill=skill
    )

with driver.session() as session:
    for i, row in jobs.iterrows():
        print(f"Processing row {i}")
        job_title = row["clean_title"]
        experience_level = row["final_level"]
        skill = row["Skill_List"]
        session.execute_write(create_job_graph, job_title, experience_level, skill)

driver.close()